In [1]:
# 00_setup — Cell 1: environment and package versions
import sys, platform, importlib

print("Python     :", sys.version.split()[0])
print("Executable :", sys.executable)
print("OS         :", platform.platform())
print()
mods = ["pandas", "numpy", "pyarrow", "duckdb", "polars", "lightgbm", "sklearn",
        "pyod", "shap", "mlxtend", "matplotlib", "seaborn", "networkx"]
for m in mods:
    v = getattr(importlib.import_module(m), "__version__", "?")
    print(f"{m:12s} {v}")

Python     : 3.11.16
Executable : D:\conda\envs\m1\python.exe
OS         : Windows-10-10.0.26200-SP0

pandas       2.3.3
numpy        2.4.6
pyarrow      25.0.1
duckdb       1.5.5
polars       1.44.2
lightgbm     4.7.0
sklearn      1.9.1
pyod         3.6.6
shap         0.51.0
mlxtend      0.25.0
matplotlib   3.11.2
seaborn      0.13.2
networkx     3.6.1


In [2]:
# 00_setup — Cell 2: project paths and free disk space
from pathlib import Path
import shutil

PATHS = {
    "raw":      Path("D:/m1/raw"),       # original Kaggle downloads, never edited
    "lake":     Path("D:/m1/lake"),      # clean Parquet archive (HDD)
    "work":     Path("C:/m1/work"),      # working Parquet + DuckDB file (fast SSD)
    "duck_tmp": Path("C:/m1/duck_tmp"),  # DuckDB spill directory
    "repo":     Path("C:/m1/repo"),      # git repo, code only
}
for name, p in PATHS.items():
    p.mkdir(parents=True, exist_ok=True)
    print(f"{name:9s} {str(p):16s} exists={p.exists()}")

print()
for drive in ["C:/", "D:/"]:
    total, used, free = shutil.disk_usage(drive)
    print(f"{drive}  free = {free/1e9:6.1f} GB  of  {total/1e9:6.1f} GB")

raw       D:\m1\raw        exists=True
lake      D:\m1\lake       exists=True
work      C:\m1\work       exists=True
duck_tmp  C:\m1\duck_tmp   exists=True
repo      C:\m1\repo       exists=True

C:/  free =   48.7 GB  of   253.7 GB
D:/  free =  853.8 GB  of  1000.2 GB


In [3]:
# 00_setup — Cell 3: DuckDB connection with memory, thread and spill settings
import duckdb, os

DB_FILE = PATHS["work"] / "m1.duckdb"
con = duckdb.connect(str(DB_FILE))
con.execute("SET memory_limit = '9GB'")
con.execute("SET threads = 8")
con.execute("SET temp_directory = 'C:/m1/duck_tmp'")
con.execute("SET max_temp_directory_size = '20GB'")

print("DuckDB", con.execute("SELECT version()").fetchone()[0])
for k in ["memory_limit", "threads", "temp_directory", "max_temp_directory_size"]:
    print(f"{k:24s}", con.execute(f"SELECT current_setting('{k}')").fetchone()[0])

print(con.execute("SELECT 40 + 2 AS answer").fetchdf())   # sanity query
con.close()
print("DB file:", DB_FILE, "| size:", os.path.getsize(DB_FILE), "bytes")

DuckDB v1.5.5
memory_limit             8.3 GiB
threads                  8
temp_directory           C:/m1/duck_tmp
max_temp_directory_size  18.6 GiB
   answer
0      42
DB file: C:\m1\work\m1.duckdb | size: 12288 bytes


In [1]:
# 00_setup — Cell 4: verify the shared config module used by all later notebooks
import sys; sys.path.insert(0, "..")
from src.config import PATHS, SEED, DUCKDB_FILE, duckdb_connect, Timer

print("SEED        =", SEED)
print("DUCKDB_FILE =", DUCKDB_FILE)
with Timer("config smoke test"):
    con = duckdb_connect()
    print("memory_limit =", con.execute("SELECT current_setting('memory_limit')").fetchone()[0])
    con.close()

SEED        = 42
DUCKDB_FILE = C:\m1\work\m1.duckdb
memory_limit = 8.3 GiB
[config smoke test] wall = 1.6 s | RSS now = 0.07 GB | peak RSS = 0.08 GB
